In [34]:
from ultralytics import YOLO
import torch

torch.nn.Conv2d

# Downloads automatically from Ultralytics on first run, then cached in ~/.cache/ultralytics
# Creates instance of the YOLO wrapper class, which loads the model
model_name = 'yolov8n.pt'

model = YOLO(model_name)
model.info()

# model.model is the actual nn.Module (DetectionModel) inside the YOLO wrapper
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

inner_model = model.model.to(device)
print(f'Inner model type: {type(inner_model).__name__}')

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
Device: cpu
Inner model type: DetectionModel


In [35]:
from pathlib import Path
import urllib
import zipfile
import json as _json

N_CALIBRATION = 64   # paper uses 64 calibration samples

def get_calibration_images(n: int = N_CALIBRATION) -> list:
    """
    Return a list of image file paths for calibration.

    Priority:
      1. Local cocosample/ folder (teammate's original approach)
      2. Download n images from COCO val2017
    """
    local = Path('cocosample')
    if local.exists() and len(list(local.glob('*.jpg'))) > 0:
        imgs = sorted(local.glob('*.jpg'))[:n]
        print(f'Using local cocosample/ — found {len(imgs)} images')
        return [str(p.resolve()) for p in imgs]

    print('cocosample/ not found — downloading COCO val2017 images...')
    download_dir = Path('coco_calib')
    download_dir.mkdir(exist_ok=True)

    base_url = 'http://images.cocodataset.org/val2017/'
    ann_url  = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    ann_dir  = Path('coco_annotations')

    if not ann_dir.exists():
        print('  Downloading COCO annotations (~250MB, done once)...')
        ann_zip = Path('annotations.zip')
        urllib.request.urlretrieve(ann_url, ann_zip)
        with zipfile.ZipFile(ann_zip, 'r') as z:
            z.extract('annotations/instances_val2017.json', 'coco_annotations')
        ann_zip.unlink()

    with open('coco_annotations/annotations/instances_val2017.json') as f:
        coco = _json.load(f)

    image_files = []
    for img_info in coco['images'][:n]:
        fname = img_info['file_name']
        fpath = download_dir / fname
        if not fpath.exists():
            urllib.request.urlretrieve(base_url + fname, fpath)
        image_files.append(str(fpath.resolve()))

    print(f'  Downloaded {len(image_files)} calibration images to coco_calib/')
    return image_files


calibration = get_calibration_images(N_CALIBRATION)
print(f'Calibration set: {len(calibration)} images')

# Map to save layer activations
activations = {}

def get_activations(name):
    def hook(module, input, output):
        # Handle tensor output or tuple/list output
        if isinstance(output, torch.Tensor):
            out = output
        elif isinstance(output, (tuple, list)) and len(output) > 0:
            out = output[0]
        else:
            return
        if isinstance(out, torch.Tensor):
            activations[name] = out.detach().cpu()
    return hook

handles = []

# FIX: register hooks on model.model (the inner nn.Module)
for name, layer in model.model.named_modules():
    if len(list(layer.children())) == 0:
        handles.append(layer.register_forward_hook(get_activations(name)))

print(f'Registered {len(handles)} hooks on model.model')

cocosample/ not found — downloading COCO val2017 images...
  Downloaded 64 calibration images to coco_calib/
Calibration set: 64 images
Registered 129 hooks on model.model


In [36]:
# Automatically find images — works on any machine
def get_image_paths(folder: str = "cocosample") -> list[str]:
    """
    Returns absolute paths to all jpg/png images in the given folder.
    Uses Path so it works on Windows, Mac, and Linux without changes.
    """
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(
            f"Could not find '{folder}'. "
            f"Current directory is: {os.getcwd()}"
        )
    paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
    if not paths:
        raise FileNotFoundError(f"No images found in '{folder}'")
    return [str(p) for p in paths]

calibration_paths = get_image_paths("coco_calib")

# Run inference
for img_path in calibration_paths:
    with torch.no_grad():
        model(img_path)   # pass full path directly — no string concatenation

for h in handles:
    h.remove()

image 1/1 /Users/hypero/COMP6258-Coursework/yoloDGQ/coco_calib/000000006818.jpg: 640x448 1 toilet, 52.4ms
Speed: 0.9ms preprocess, 52.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /Users/hypero/COMP6258-Coursework/yoloDGQ/coco_calib/000000016228.jpg: 448x640 12 persons, 1 bench, 1 horse, 1 umbrella, 45.4ms
Speed: 0.8ms preprocess, 45.4ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 /Users/hypero/COMP6258-Coursework/yoloDGQ/coco_calib/000000017627.jpg: 480x640 3 persons, 10 cars, 49.0ms
Speed: 0.8ms preprocess, 49.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/hypero/COMP6258-Coursework/yoloDGQ/coco_calib/000000025560.jpg: 480x640 1 person, 1 cat, 1 cup, 1 tv, 52.8ms
Speed: 0.9ms preprocess, 52.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/hypero/COMP6258-Coursework/yoloDGQ/coco_calib/000000037777.jpg: 448x640 6 oranges, 1 dining table, 1 oven

In [ ]:
from sklearn.cluster import KMeans
from tqdm import tqdm
import json

n_groups = 2
n_bits   = 8

QUANT_PARAMS = {}

for name, layer in tqdm(model.model.named_modules(), desc="Processing layers"):
    if len(list(layer.children())) == 0:
        if isinstance(layer, torch.nn.Conv2d):

            layer_acts   = activations[name].squeeze(0)
            channels_dim = layer_acts.view(layer_acts.shape[0], -1)

            if channels_dim.shape[0] < n_groups:
                print(f"  Skipping {name} — only {channels_dim.shape[0]} channels, less than n_groups={n_groups}")
                continue
            
            kmeans = KMeans(n_clusters=n_groups, random_state=0, n_init='auto')
            kmeans.fit(channels_dim)
            groups = kmeans.predict(channels_dim)

            # Build per-group min/max — logic unchanged from original
            quant_groups = {}
            for i in range(len(groups)):
                g = groups[i]
                if g in quant_groups:
                    if max(channels_dim[i]) > quant_groups[g]['max']:
                        quant_groups[g]['max'] = max(channels_dim[i])
                    if min(channels_dim[i]) < quant_groups[g]['min']:
                        quant_groups[g]['min'] = min(channels_dim[i])
                else:
                    quant_groups[g] = {'min': min(channels_dim[i]), 'max': max(channels_dim[i])}

            quant_params = {}
            for group, params in quant_groups.items():
                q_s = (params['max'] - params['min']) / (pow(2, n_bits)-1)
                z   = max(0, min(pow(2, n_bits)-1, torch.round(-params['min'] / q_s)))
                quant_params[group] = {'q_s': q_s.detach().item(), 'z': z.detach().item()}

            channel_quant = dict(zip(range(len(channels_dim)), [quant_params[i] for i in groups]))

            QUANT_PARAMS[name] = str(channel_quant)


with open(f"{model_name.replace('.pt', '')}_quant_params.json", 'w+') as fp:
    json.dump(QUANT_PARAMS, fp)


Processing layers: 168it [00:40,  4.11it/s]

  Skipping model.22.dfl.conv — only 1 channels, less than n_groups=2
